In [1]:
import math
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ibtracs.NI.list.v04r00.csv")

# Build descriptive column headers from header rows
column_names_with_desc = []
for col, desc in zip(df.columns, df.iloc[0]):
    if pd.isnull(df.iloc[1][col]):
        column_names_with_desc.append(col)
    else:
        column_names_with_desc.append(f"{col}_{desc}")

df = df.iloc[1:].copy()
df.columns = column_names_with_desc

# Filter North Indian Ocean basin
df_ni = df[df['BASIN_ '] == 'NI'].copy()
df = df_ni

/tmp/ipykernel_3132/2434374005.py:1: DtypeWarning: Columns (1,2,8,9,14,161,162) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/ibtracs.NI.list.v04r00.csv")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
prefixes_to_delete = ['WMO', 'USA', 'TOKYO', 'CMA', 'HKO', 'REUNION', 'BOM', 'NADI', 'WELLINGTON', 'DS', 'TD', 'NEUMANN', 'MLC', 'NEWDELHI', 'IFLAG']
columns_to_keep = [col for col in df.columns if not any(col.startswith(prefix) for prefix in prefixes_to_delete)]
df_filtered = df[columns_to_keep].copy()

df_filtered['SEASON_Year'] = pd.to_numeric(df_filtered['SEASON_Year'], errors='coerce')
df_filtered = df_filtered[df_filtered['SEASON_Year'] >= 1900].copy()

In [ ]:
def angle_from_coordinates(lat1, lon1, lat2, lon2):
    d_lon = math.radians(lon2 - lon1)
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)

    y = math.sin(d_lon) * math.cos(lat2_rad)
    x = math.cos(lat1_rad) * math.sin(lat2_rad) - math.sin(lat1_rad) * math.cos(lat2_rad) * math.cos(d_lon)
    brng = math.atan2(y, x)
    brng = math.degrees(brng)
    return (360 - ((brng + 360) % 360)) % 360

def calculate_angles(df):
    angles = []
    prev_lat = None
    prev_lon = None
    prev_sid = None

    for _, row in df.iterrows():
        curr_lat = row['LAT_degrees_north']
        curr_lon = row['LON_degrees_east']
        curr_sid = row['NAME_ ']

        if prev_lat is None or prev_lon is None or curr_sid != prev_sid:
            angles.append(0.0)
        else:
            angles.append(angle_from_coordinates(prev_lat, prev_lon, curr_lat, curr_lon))

        prev_lat, prev_lon, prev_sid = curr_lat, curr_lon, curr_sid

    df['ANGLE'] = angles
    return df

# Convert target coordinates to float for distance/angle calculations
df_filtered['LAT_degrees_north'] = pd.to_numeric(df_filtered['LAT_degrees_north'], errors='coerce').fillna(0)
df_filtered['LON_degrees_east'] = pd.to_numeric(df_filtered['LON_degrees_east'], errors='coerce').fillna(0)

df_filtered = calculate_angles(df_filtered)
df_filtered.reset_index(drop=True, inplace=True)

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def calculate_distances(df):
    distances = []
    prev_lat, prev_lon, prev_sid = None, None, None

    for _, row in df.iterrows():
        curr_lat = row['LAT_degrees_north']
        curr_lon = row['LON_degrees_east']
        curr_sid = row['NAME_ ']

        if prev_lat is None or prev_lon is None or curr_sid != prev_sid:
            distances.append(0.0)
        else:
            distances.append(haversine(prev_lat, prev_lon, curr_lat, curr_lon))

        prev_lat, prev_lon, prev_sid = curr_lat, curr_lon, curr_sid

    df['DISTANCE_km'] = distances
    return df

df_filtered = calculate_distances(df_filtered)

In [ ]:
df_filtered['ISO_TIME_ '] = pd.to_datetime(df_filtered['ISO_TIME_ '], errors='coerce')
df_sorted = df_filtered.sort_values(by=['SID_ ', 'ISO_TIME_ ']).copy()

# Calculate elapsed hours from storm initiation
df_sorted['TIME_DIFFERENCE_hours'] = df_sorted.groupby('SID_ ')['ISO_TIME_ '].transform(
    lambda x: (x - x.min()).dt.total_seconds() / 3600.0
)

In [ ]:
# Categorical encodings
subbasin_mapping = {'BB': 1, 'AS': 2}
nature_mapping = {'DS': 1, 'TS': 2, 'ET': 3, 'SS': 4, 'NR': 5, 'MX': 6}

df_sorted['SUBBASIN_ '] = df_sorted['SUBBASIN_ '].map(subbasin_mapping).fillna(0)
df_sorted['NATURE_ '] = df_sorted['NATURE_ '].map(nature_mapping).fillna(0)

# Datetime feature extraction
df_sorted['Hour_of_the_Day'] = df_sorted['ISO_TIME_ '].dt.hour
df_sorted['Day_of_the_Week'] = df_sorted['ISO_TIME_ '].dt.dayofweek
df_sorted['Month'] = df_sorted['ISO_TIME_ '].dt.month
df_sorted['Season'] = df_sorted['ISO_TIME_ '].dt.month // 4

In [ ]:
numeric_columns = [
    'DIST2LAND_km', 'LANDFALL_km', 'NUMBER_ ',
    'STORM_SPEED_kts', 'STORM_DIR_degrees', 'Hour_of_the_Day',
    'Day_of_the_Week', 'Month', 'Season', 'ANGLE', 'DISTANCE_km', 'TIME_DIFFERENCE_hours'
]

for col in numeric_columns:
    df_sorted[col] = pd.to_numeric(df_sorted[col], errors='coerce').fillna(0).astype(float)

In [ ]:
# Features excluding target columns (Latitude and Longitude)
selected_features = [
    'NUMBER_ ', 'SUBBASIN_ ', 'DIST2LAND_km', 'STORM_SPEED_kts',
    'LANDFALL_km', 'STORM_DIR_degrees', 'ANGLE', 'DISTANCE_km',
    'TIME_DIFFERENCE_hours', 'Month', 'Season'
]

X = df_sorted[selected_features]
y_lat = df_sorted['LAT_degrees_north']
y_lon = df_sorted['LON_degrees_east']

# Split target variables synchronously using a set random state
X_train, X_test, y_train_lat, y_test_lat = train_test_split(X, y_lat, test_size=0.2, random_state=42)
_, _, y_train_lon, y_test_lon = train_test_split(X, y_lon, test_size=0.2, random_state=42)

In [ ]:
linear_reg_lat = LinearRegression()
linear_reg_lat.fit(X_train, y_train_lat)

linear_reg_lon = LinearRegression()
linear_reg_lon.fit(X_train, y_train_lon)

y_pred_lat = linear_reg_lat.predict(X_test)
y_pred_lon = linear_reg_lon.predict(X_test)

rmse_lat = np.sqrt(mean_squared_error(y_test_lat, y_pred_lat))
rmse_lon = np.sqrt(mean_squared_error(y_test_lon, y_pred_lon))
r2_lat = r2_score(y_test_lat, y_pred_lat)
r2_lon = r2_score(y_test_lon, y_pred_lon)

print(f"--- Model Accuracy Metrics ---")
print(f"Latitude Prediction Accuracy (R2 Score): {r2_lat:.4f}")
print(f"Longitude Prediction Accuracy (R2 Score): {r2_lon:.4f}")
print(f"\n--- Error Metrics ---")
print(f"Latitude RMSE: {rmse_lat:.4f}")
print(f"Longitude RMSE: {rmse_lon:.4f}")

--- Model Accuracy Metrics ---
Latitude Prediction Accuracy (R2 Score): 0.4409
Longitude Prediction Accuracy (R2 Score): 0.7509

--- Error Metrics ---
Latitude RMSE: 4.4887
Longitude RMSE: 4.8338


In [ ]:
print(f"--- Model Accuracy in Percentage ---")
print(f"Latitude Prediction Accuracy: {r2_lat * 100:.2f}%")
print(f"Longitude Prediction Accuracy: {r2_lon * 100:.2f}%")

--- Model Accuracy in Percentage ---
Latitude Prediction Accuracy: 44.09%
Longitude Prediction Accuracy: 75.09%


In [ ]:
joblib.dump(linear_reg_lat, 'linear_reg_lat_model.joblib')
joblib.dump(linear_reg_lon, 'linear_reg_lon_model.joblib')

['linear_reg_lon_model.joblib']